# NEC VN v2 (P0 + P1 + P2)

Notebook gon de chay 3 setup P0/P1/P2 voi dual-network (CalibNet + EmuNet).
Phuong trinh f2 da sua dung dau theo Model (3): +muh*H.

In [4]:
import os
import copy
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.autograd import grad
import matplotlib.pyplot as plt

SEED = 1234

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(SEED)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device =', device)

REGION = 'vn'
TRAIN_PATH = 'data/vn_train.csv'
VAL_PATH = 'data/vn_val.csv'


def parse_pair_matrix(raw):
    if raw.ndim != 2:
        raise ValueError(f'Expected 2D matrix, got {raw.shape}')
    if raw.shape[0] == 2:
        return raw[0].astype(np.float32), raw[1].astype(np.float32)
    if raw.shape[1] == 2:
        return raw[:, 0].astype(np.float32), raw[:, 1].astype(np.float32)
    raise ValueError(f'Unsupported shape: {raw.shape}')


def load_split(train_path, val_path):
    tr = np.genfromtxt(train_path, delimiter=',')
    va = np.genfromtxt(val_path, delimiter=',')
    t_train, y_train = parse_pair_matrix(tr)
    t_val, y_val = parse_pair_matrix(va)

    idx_tr = np.argsort(t_train)
    idx_va = np.argsort(t_val)
    t_train, y_train = t_train[idx_tr], y_train[idx_tr]
    t_val, y_val = t_val[idx_va], y_val[idx_va]

    print(f'Train: n={len(t_train)}, weeks=[{t_train.min():.0f},{t_train.max():.0f}], y=[{y_train.min():.0f},{y_train.max():.0f}]')
    print(f'Val:   n={len(t_val)}, weeks=[{t_val.min():.0f},{t_val.max():.0f}], y=[{y_val.min():.0f},{y_val.max():.0f}]')
    return t_train.astype(np.float32), y_train.astype(np.float32), t_val.astype(np.float32), y_val.astype(np.float32)


def build_calib_features(y_train, dim=10):
    y_log = np.log1p(y_train)
    idx = np.linspace(0, len(y_log) - 1, dim).astype(int)
    feat = y_log[idx]
    feat = (feat - feat.mean()) / (feat.std() + 1e-6)
    return feat.astype(np.float32)


t_train, y_train, t_val, y_val = load_split(TRAIN_PATH, VAL_PATH)


device = cpu
Train: n=80, weeks=[1,80], y=[1021,178131]
Val:   n=10, weeks=[81,90], y=[180437,208146]


In [5]:
param_bounds = {
    'lambda': (0.0, 0.6),
    'betam': (0.0, 0.05),
    'mum': (0.001, 0.016),
    'thetam': (0.01, 0.2),
    'muh': (0.001, 0.0195),
    'betah': (0.001, 0.055),
    'thetah': (0.01, 0.53),
    'gammah': (0.05, 0.1),
}
CALIB_INPUT_DIM = 10


class CalibNet(nn.Module):
    def __init__(self, bounds):
        super().__init__()
        self.fc1 = nn.Linear(CALIB_INPUT_DIM, 64)
        self.fc2 = nn.Linear(64, 64)
        self.fc3 = nn.Linear(64, len(bounds))
        self.bounds = bounds
        self.names = list(bounds.keys())

    def forward(self, x):
        h = F.gelu(self.fc1(x))
        h = F.gelu(self.fc2(h))
        raw = self.fc3(h)
        params = {}
        for i, name in enumerate(self.names):
            lo, hi = self.bounds[name]
            params[name] = torch.tanh(raw[..., i]) * (hi - lo) / 2 + (hi + lo) / 2
        return params


class EmuNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.ModuleList([
            nn.Linear(1, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 32),
            nn.Linear(32, 32),
        ])
        self.out = nn.Linear(32, 8)

    def forward(self, t_batch):
        h = t_batch
        for layer in self.layers:
            h = F.gelu(layer(h))
        return F.softplus(self.out(h))


class NECDual(nn.Module):
    def __init__(self, t_train, y_train, calib_features=None):
        super().__init__()
        self.device = device
        self.t = torch.tensor(t_train, dtype=torch.float32, requires_grad=True, device=self.device)
        self.t_batch = self.t.reshape(-1, 1)
        self.y = torch.tensor(y_train, dtype=torch.float32, device=self.device)
        self.y_log = torch.log1p(self.y)
        self.t_max = float(t_train[-1])

        self.calib = CalibNet(param_bounds).to(self.device)
        self.emu = EmuNet().to(self.device)

        if calib_features is None:
            self.calib_input = torch.zeros(1, CALIB_INPUT_DIM, dtype=torch.float32, device=self.device)
        else:
            self.calib_input = torch.tensor(calib_features, dtype=torch.float32, device=self.device).reshape(1, -1)

        self.log_var_data = nn.Parameter(torch.zeros(1, device=self.device))
        self.log_var_phys = nn.Parameter(torch.zeros(1, device=self.device))

    def ode_residuals(self, params, t, t_batch):
        lam = params['lambda']
        betam = params['betam']
        mum = params['mum']
        thetam = params['thetam']
        muh = params['muh']
        betah = params['betah']
        thetah = params['thetah']
        gammah = params['gammah']

        y_hat = self.emu(t_batch)
        Hr_hat = y_hat[:, 0:1]
        Hs_hat = y_hat[:, 1:2]
        He_hat = y_hat[:, 2:3]
        Hi_hat = y_hat[:, 3:4]
        Me_hat = y_hat[:, 4:5]
        Ms_hat = y_hat[:, 5:6]
        Mi_hat = y_hat[:, 6:7]
        Itot_hat = y_hat[:, 7:8]

        Hr_t = grad(Hr_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Hs_t = grad(Hs_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        He_t = grad(He_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Hi_t = grad(Hi_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Me_t = grad(Me_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Ms_t = grad(Ms_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Mi_t = grad(Mi_hat.sum(), t, retain_graph=True, create_graph=True)[0]
        Itot_t = grad(Itot_hat.sum(), t, retain_graph=True, create_graph=True)[0]

        Hr = torch.expm1(Hr_hat.squeeze(1))
        Hs = torch.expm1(Hs_hat.squeeze(1))
        He = torch.expm1(He_hat.squeeze(1))
        Hi = torch.expm1(Hi_hat.squeeze(1))
        Me = torch.expm1(Me_hat.squeeze(1))
        Ms = torch.expm1(Ms_hat.squeeze(1))
        Mi = torch.expm1(Mi_hat.squeeze(1))
        Itot = torch.expm1(Itot_hat.squeeze(1))

        H = torch.clamp(Hr + Hs + He + Hi, min=1e-8)
        M = torch.clamp(Me + Ms + Mi, min=1e-8)

        f1 = Hr_t - (gammah * Hi - muh * Hr) / (1.0 + Hr)
        # f2 sign fixed according to paper Model (3)
        f2 = Hs_t - (muh * H - betah * Mi * Hs / M - muh * Hs) / (1.0 + Hs)
        f3 = He_t - (betah * Mi * Hs / M - thetah * He - muh * He) / (1.0 + He)
        f4 = Hi_t - (thetah * He - gammah * Hi - muh * Hi) / (1.0 + Hi)
        f5 = Me_t - (betam * Hi * Ms / H - thetam * Me - mum * Me) / (1.0 + Me)
        f6 = Ms_t - (lam - betam * Hi * Ms / H - mum * Ms) / (1.0 + Ms)
        f7 = Mi_t - (thetam * Me - mum * Mi) / (1.0 + Mi)
        f8 = Itot_t - (thetah * He) / (1.0 + Itot)

        return f1, f2, f3, f4, f5, f6, f7, f8, Itot_hat.squeeze(1)

    def netf_data(self):
        params = self.calib(self.calib_input)
        return self.ode_residuals(params, self.t, self.t_batch)

    def netf_colloc(self, n_colloc=160):
        t_col = torch.rand(n_colloc, device=self.device) * self.t_max
        t_col.requires_grad_(True)
        t_col_batch = t_col.reshape(-1, 1)
        params = self.calib(self.calib_input)
        out = self.ode_residuals(params, t_col, t_col_batch)
        return out[:8]

    def predict_itot_numpy(self, t_np):
        t_tensor = torch.tensor(t_np, dtype=torch.float32, device=self.device).reshape(-1, 1)
        with torch.no_grad():
            y_log = self.emu(t_tensor)[:, 7]
            y = torch.expm1(y_log)
        return y.detach().cpu().numpy()


def rel_rmse(y_true, y_pred):
    den = np.sum(np.square(y_true))
    if den <= 1e-12:
        return np.nan
    return float(np.sqrt(np.sum((y_true - y_pred) ** 2) / den))


def quantile_cap(values, q=0.9, mult=1.2, floor=1e-6):
    arr = np.asarray(values, dtype=np.float32)
    if arr.size == 0:
        return float(floor)
    return float(max(np.quantile(arr, q) * mult, floor))


def estimate_tail_caps(y_train, cfg):
    dy = np.diff(y_train)
    if len(dy) == 0:
        return 1.0, 1e-3

    growth = dy / (y_train[:-1] + 1.0)
    tail_w = int(cfg.get('cap_tail_window', 20))
    tail_w = max(4, min(tail_w, len(dy)))

    dy_tail = dy[-tail_w:]
    g_tail = growth[-tail_w:]

    dy_cap_tail = quantile_cap(
        dy_tail[dy_tail > 0.0],
        q=float(cfg.get('spike_cap_q_tail', 0.95)),
        mult=float(cfg.get('spike_cap_mult_tail', 1.8)),
        floor=1.0,
    )
    g_cap_tail = quantile_cap(
        g_tail[g_tail > 0.0],
        q=float(cfg.get('growth_cap_q_tail', 0.95)),
        mult=float(cfg.get('growth_cap_mult_tail', 2.0)),
        floor=1e-4,
    )

    dy_cap_global = quantile_cap(
        dy[dy > 0.0],
        q=float(cfg.get('spike_cap_q_global', 0.80)),
        mult=float(cfg.get('spike_cap_mult_global', 0.70)),
        floor=1.0,
    )
    g_cap_global = quantile_cap(
        growth[growth > 0.0],
        q=float(cfg.get('growth_cap_q_global', 0.80)),
        mult=float(cfg.get('growth_cap_mult_global', 0.40)),
        floor=1e-4,
    )

    dy_cap = max(dy_cap_tail, dy_cap_global)
    growth_cap = max(g_cap_tail, g_cap_global)
    return float(dy_cap), float(growth_cap)


def project_sequence(y_seq, y0, dy_cap, growth_cap, smooth_alpha=0.0):
    y = np.asarray(y_seq, dtype=np.float64)
    out = np.zeros_like(y)
    prev = max(float(y0), 0.0)
    alpha = float(np.clip(smooth_alpha, 0.0, 0.95))

    for i in range(len(y)):
        raw = max(float(y[i]), 0.0)
        max_step = min(float(dy_cap), float(growth_cap) * (prev + 1.0))
        max_step = max(max_step, 0.0)
        lo = prev
        hi = prev + max_step
        clipped = min(max(raw, lo), hi)
        if i > 0 and alpha > 0.0:
            clipped = alpha * out[i - 1] + (1.0 - alpha) * clipped
        out[i] = clipped
        prev = clipped

    return out.astype(np.float32)


def project_sequence_torch(y_seq, y0, dy_cap, growth_cap, smooth_alpha=0.0):
    y = y_seq.reshape(-1)
    device_local = y.device
    dtype_local = y.dtype

    prev = torch.clamp(torch.as_tensor(y0, device=device_local, dtype=dtype_local), min=0.0)
    dy_cap_t = torch.as_tensor(float(dy_cap), device=device_local, dtype=dtype_local)
    growth_cap_t = torch.as_tensor(float(growth_cap), device=device_local, dtype=dtype_local)
    alpha = float(np.clip(smooth_alpha, 0.0, 0.95))

    out = []
    for i in range(y.shape[0]):
        raw = torch.clamp(y[i], min=0.0)
        max_step = torch.minimum(dy_cap_t, growth_cap_t * (prev + 1.0))
        max_step = torch.clamp(max_step, min=0.0)
        clipped = torch.clamp(raw, min=prev, max=prev + max_step)
        if i > 0 and alpha > 0.0:
            clipped = alpha * out[-1] + (1.0 - alpha) * clipped
        out.append(clipped)
        prev = clipped

    return torch.stack(out)


def compute_phys_weights(loss_list, ema_state, cfg):
    vals = torch.stack([l.detach() for l in loss_list])
    if ema_state is None:
        ema_state = vals
    else:
        beta = float(cfg.get('phys_ema_beta', 0.95))
        ema_state = beta * ema_state + (1.0 - beta) * vals

    mode = cfg.get('phys_balance', 'inv_softmax')
    if mode == 'inv_softmax':
        eps = float(cfg.get('phys_eps', 1e-8))
        inv = 1.0 / torch.clamp(ema_state, min=eps)
        inv = inv / torch.clamp(inv.mean(), min=eps)
        clip_max = float(cfg.get('phys_weight_clip', 10.0))
        inv = torch.clamp(inv, max=clip_max)
        temp = max(float(cfg.get('phys_temp', 1.0)), 1e-6)
        logits = inv / temp
        weights = torch.softmax(logits, dim=0) * len(loss_list)
    else:
        weights = torch.ones_like(ema_state)

    return weights.detach(), ema_state


def ramp_value(epoch, start_epoch, ramp_epochs, max_val):
    if epoch < start_epoch:
        return 0.0
    if epoch < start_epoch + max(1, ramp_epochs):
        return float(max_val) * (epoch - start_epoch) / float(max(1, ramp_epochs))
    return float(max_val)


def run_experiment(name, cfg, t_train, y_train, t_val, y_val):
    seed_everything(SEED)
    features = build_calib_features(y_train, 10) if cfg['use_features'] else None
    model = NECDual(t_train, y_train, calib_features=features).to(device)

    params = list(model.calib.parameters()) + list(model.emu.parameters())
    if cfg['adaptive_weights']:
        params += [model.log_var_data, model.log_var_phys]

    optimizer = optim.Adam(params, lr=cfg['lr'], betas=(0.9, 0.999), weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.7, patience=450, min_lr=1e-7
    )

    for p in model.calib.parameters():
        p.requires_grad = False

    dy_cap, growth_cap = estimate_tail_caps(y_train, cfg)
    dy_cap_t = torch.tensor(dy_cap, dtype=torch.float32, device=device)
    growth_cap_t = torch.tensor(growth_cap, dtype=torch.float32, device=device)

    eval_horizon = int(cfg.get('eval_horizon', cfg.get('spike_horizon', 10)))
    t_future_eval_np = np.arange(float(t_train[-1]) + 1.0, float(t_train[-1]) + 1.0 + eval_horizon, dtype=np.float32)
    t_future_eval_t = torch.tensor(t_future_eval_np, dtype=torch.float32, device=device).reshape(-1, 1)

    anchor_w = int(cfg.get('anchor_tail_window', 12))
    anchor_w = max(4, min(anchor_w, len(y_train) - 1))
    dy_anchor = np.diff(y_train[-(anchor_w + 1):])
    anchor_step = float(np.median(dy_anchor[-min(6, len(dy_anchor)) :])) if len(dy_anchor) > 0 else 0.0
    y_anchor0 = float(y_train[-1] + max(anchor_step, 0.0))
    y_anchor0_t = torch.tensor(y_anchor0, dtype=torch.float32, device=device)

    n_train = len(y_train)
    tw_start = float(cfg.get('time_weight_start', 0.35))
    time_w = torch.linspace(tw_start, 1.0, n_train, device=device)
    time_w = time_w / torch.clamp(time_w.mean(), min=1e-6)

    best_score = float('inf')
    best_state = None
    best_epoch = -1
    no_improve = 0

    ema_phys = None
    start = time.time()

    for epoch in range(cfg['epochs'] + 1):
        if epoch == cfg['warmup']:
            for p in model.calib.parameters():
                p.requires_grad = True

        if epoch < cfg['warmup']:
            lambda_phys = 0.0
        elif epoch < cfg['warmup'] + cfg['ramp']:
            lambda_phys = (epoch - cfg['warmup']) / max(1, cfg['ramp']) * cfg['lambda_phys_max']
        else:
            lambda_phys = cfg['lambda_phys_max']

        proj_start = int(cfg.get('proj_start_epoch', max(200, cfg['warmup'] // 2)))
        proj_ramp = int(cfg.get('proj_ramp_epochs', 600))
        lambda_proj = ramp_value(epoch, proj_start, proj_ramp, cfg.get('lambda_proj', 0.0))

        roll_start = int(cfg.get('rollout_start_epoch', cfg['warmup']))
        roll_ramp = int(cfg.get('rollout_ramp_epochs', 600))
        lambda_roll = ramp_value(epoch, roll_start, roll_ramp, cfg.get('lambda_roll', 0.0))

        if cfg.get('use_v2_rollout', False):
            h_min = int(cfg.get('proj_horizon_min', 6))
            h_max = int(cfg.get('proj_horizon_max', eval_horizon))
            prog = max(0.0, min(1.0, (epoch - cfg['warmup']) / max(1, cfg['epochs'] - cfg['warmup'])))
            train_horizon = int(round(h_min + (h_max - h_min) * prog))
        else:
            train_horizon = int(cfg.get('spike_horizon', eval_horizon))
        train_horizon = max(4, train_horizon)

        t_future_train_np = np.arange(float(t_train[-1]) + 1.0, float(t_train[-1]) + 1.0 + train_horizon, dtype=np.float32)
        t_future_train_t = torch.tensor(t_future_train_np, dtype=torch.float32, device=device).reshape(-1, 1)

        optimizer.zero_grad()
        out_data = model.netf_data()

        err_log = out_data[8] - model.y_log
        if cfg['data_loss'] == 'huber':
            beta = float(cfg.get('huber_beta', 0.5))
            ae = torch.abs(err_log)
            loss_log_vec = torch.where(ae <= beta, 0.5 * (ae ** 2) / beta, ae - 0.5 * beta)
            loss_data_log = torch.mean(time_w * loss_log_vec)
        else:
            loss_data_log = torch.mean(time_w * (err_log ** 2))

        y_pred_train = torch.expm1(out_data[8])
        rel_err = (y_pred_train - model.y) / torch.clamp(model.y + 500.0, min=1.0)
        loss_data_rel = torch.mean(time_w * (rel_err ** 2))

        dy_pred = y_pred_train[1:] - y_pred_train[:-1]
        dy_true = model.y[1:] - model.y[:-1]
        k_tail = int(cfg.get('trend_tail_k', 16))
        k_tail = max(4, min(k_tail, dy_pred.shape[0]))
        tail_scale = torch.clamp(torch.mean(torch.abs(dy_true[-k_tail:])), min=1.0)
        loss_trend = F.smooth_l1_loss(
            dy_pred[-k_tail:], dy_true[-k_tail:], beta=float(cfg.get('trend_beta', 1200.0))
        ) / tail_scale

        loss_data = (
            loss_data_log
            + float(cfg.get('lambda_data_rel', 0.10)) * loss_data_rel
            + float(cfg.get('lambda_trend', 0.45)) * loss_trend
        )

        phys_data = [F.mse_loss(r, torch.zeros_like(r)) for r in out_data[:8]]
        if cfg['use_colloc']:
            out_col = model.netf_colloc(n_colloc=cfg['n_colloc'])
            phys_col = [F.mse_loss(r, torch.zeros_like(r)) for r in out_col]
            phys_terms = [0.5 * ld + 0.5 * lc for ld, lc in zip(phys_data, phys_col)]
        else:
            phys_terms = phys_data

        phys_weights, ema_phys = compute_phys_weights(phys_terms, ema_phys, cfg)
        loss_phys = torch.sum(phys_weights * torch.stack(phys_terms))

        y_future = torch.expm1(model.emu(t_future_train_t)[:, 7])
        dy_future = y_future[1:] - y_future[:-1]

        if cfg['use_mono']:
            loss_mono = torch.mean(F.relu(-dy_future))
        else:
            loss_mono = torch.tensor(0.0, device=device)

        loss_spike = torch.mean((F.relu(dy_future - dy_cap_t) / torch.clamp(dy_cap_t, min=1.0)) ** 2)
        growth_future = dy_future / torch.clamp(y_future[:-1] + 1.0, min=1.0)
        loss_growth = torch.mean((F.relu(growth_future - growth_cap_t) / torch.clamp(growth_cap_t, min=1e-4)) ** 2)

        if dy_future.shape[0] > 1:
            ddy_future = dy_future[1:] - dy_future[:-1]
            scale = torch.clamp(torch.mean(y_future), min=1.0)
            loss_smooth = torch.mean((ddy_future / scale) ** 2)
        else:
            loss_smooth = torch.tensor(0.0, device=device)

        loss_anchor = ((y_future[0] - y_anchor0_t) / torch.clamp(y_anchor0_t, min=1.0)) ** 2
        loss_ic0 = ((y_pred_train[0] - model.y[0]) / torch.clamp(model.y[0], min=1.0)) ** 2

        if cfg.get('adaptive_boundary', False):
            spike_violation = torch.mean(F.relu(dy_future - dy_cap_t) / torch.clamp(dy_cap_t, min=1.0)).detach()
            growth_violation = torch.mean(F.relu(growth_future - growth_cap_t) / torch.clamp(growth_cap_t, min=1e-4)).detach()
            boundary_scale = 1.0 + float(cfg.get('ab_alpha', 1.5)) * (spike_violation + growth_violation)
            boundary_scale = float(np.clip(boundary_scale, 1.0, float(cfg.get('ab_max', 3.0))))
        else:
            boundary_scale = 1.0

        y_proj_future = project_sequence_torch(
            y_future,
            y0=float(y_train[-1]),
            dy_cap=dy_cap,
            growth_cap=growth_cap,
            smooth_alpha=float(cfg.get('proj_smooth_alpha_train', cfg.get('proj_smooth_alpha', 0.15))),
        )
        norm_proj = torch.clamp(torch.mean(y_future.detach()) + 1.0, min=1.0)
        loss_proj_cons = torch.mean(((y_future - y_proj_future) / norm_proj) ** 2)

        if cfg.get('use_v2_rollout', False):
            rollout_steps = int(cfg.get('rollout_steps', 4))
            rollout_steps = max(2, rollout_steps)
            t_roll = np.arange(float(t_train[-1]) - rollout_steps + 1.0, float(t_train[-1]) + 1.0 + train_horizon, dtype=np.float32)
            t_roll_t = torch.tensor(t_roll, dtype=torch.float32, device=device).reshape(-1, 1)
            y_roll = torch.expm1(model.emu(t_roll_t)[:, 7])
            y_roll_proj = project_sequence_torch(
                y_roll,
                y0=float(y_train[max(0, len(y_train) - rollout_steps)]),
                dy_cap=dy_cap,
                growth_cap=growth_cap,
                smooth_alpha=float(cfg.get('proj_smooth_alpha_train', cfg.get('proj_smooth_alpha', 0.15))),
            )
            y_roll_f = y_roll[rollout_steps:]
            y_roll_proj_f = y_roll_proj[rollout_steps:]
            dy_roll = y_roll_f[1:] - y_roll_f[:-1]
            dy_roll_proj = y_roll_proj_f[1:] - y_roll_proj_f[:-1]
            norm_roll = torch.clamp(torch.mean(y_roll_proj_f.detach()) + 1.0, min=1.0)
            loss_roll_lvl = torch.mean(((y_roll_f - y_roll_proj_f.detach()) / norm_roll) ** 2)
            loss_roll_dyn = torch.mean(((dy_roll - dy_roll_proj.detach()) / torch.clamp(dy_cap_t, min=1.0)) ** 2)
            loss_rollout = loss_roll_lvl + float(cfg.get('lambda_roll_dyn', 0.5)) * loss_roll_dyn
        else:
            loss_rollout = torch.tensor(0.0, device=device)

        phys_term = (
            lambda_phys * loss_phys
            + cfg['lambda_mono'] * loss_mono
            + boundary_scale * float(cfg.get('lambda_spike', 0.0)) * loss_spike
            + boundary_scale * float(cfg.get('lambda_growth', 0.0)) * loss_growth
            + float(cfg.get('lambda_smooth', 0.0)) * loss_smooth
            + boundary_scale * float(cfg.get('lambda_anchor', 0.0)) * loss_anchor
            + float(cfg.get('lambda_ic0', 0.0)) * loss_ic0
        )

        loss_obj = loss_data + phys_term + lambda_proj * loss_proj_cons + lambda_roll * loss_rollout

        if cfg['adaptive_weights']:
            lvd = torch.clamp(model.log_var_data, -6.0, 6.0)
            lvp = torch.clamp(model.log_var_phys, -6.0, 6.0)
            wd = torch.exp(-lvd)
            wp = torch.exp(-lvp)
            loss = 0.5 * wd * (loss_data + lambda_proj * loss_proj_cons + lambda_roll * loss_rollout) + 0.5 * lvd + 0.5 * wp * phys_term + 0.5 * lvp
        else:
            loss = loss_obj

        loss.backward()
        optimizer.step()
        scheduler.step(loss.detach())

        if epoch % cfg['eval_every'] == 0:
            y_val_raw = model.predict_itot_numpy(t_val)
            if cfg.get('use_projection', False):
                y_val_eval = project_sequence(
                    y_val_raw,
                    y0=float(y_train[-1]),
                    dy_cap=dy_cap,
                    growth_cap=growth_cap,
                    smooth_alpha=float(cfg.get('proj_smooth_alpha', 0.15)),
                )
            else:
                y_val_eval = y_val_raw

            val_mae = float(np.mean(np.abs(y_val - y_val_eval)))
            val_rmse = float(np.sqrt(np.mean((y_val - y_val_eval) ** 2)))

            y_fore_raw = model.predict_itot_numpy(t_future_eval_np)
            if cfg.get('use_projection', False):
                y_fore_eval = project_sequence(
                    y_fore_raw,
                    y0=float(y_train[-1]),
                    dy_cap=dy_cap,
                    growth_cap=growth_cap,
                    smooth_alpha=float(cfg.get('proj_smooth_alpha', 0.15)),
                )
            else:
                y_fore_eval = y_fore_raw

            if len(y_fore_eval) > 1:
                dy_fore = np.diff(y_fore_eval)
                max_fore_jump = float(np.max(dy_fore))
                max_fore_growth = float(np.max(dy_fore / (y_fore_eval[:-1] + 1.0)))
            else:
                max_fore_jump = 0.0
                max_fore_growth = 0.0

            spike_pen = max(0.0, max_fore_jump / (dy_cap + 1e-6) - 1.0)
            growth_pen = max(0.0, max_fore_growth / (growth_cap + 1e-8) - 1.0)
            raw_mix = float(cfg.get('best_raw_mix', 0.0))
            raw_mix = max(0.0, min(0.8, raw_mix))
            score_base = (1.0 - raw_mix) * val_rmse + raw_mix * float(np.sqrt(np.mean((y_val - y_val_raw) ** 2)))
            score = score_base * (
                1.0
                + float(cfg.get('best_spike_alpha', 0.20)) * spike_pen
                + float(cfg.get('best_growth_alpha', 0.20)) * growth_pen
            )

            select_start = int(cfg.get('best_select_start_epoch', 0))
            if epoch >= select_start:
                if score < best_score:
                    best_score = score
                    best_state = copy.deepcopy(model.state_dict())
                    best_epoch = epoch
                    no_improve = 0
                else:
                    no_improve += cfg['eval_every']

            if epoch % cfg['print_every'] == 0:
                wmin = float(torch.min(phys_weights).item())
                wmax = float(torch.max(phys_weights).item())
                print(
                    f"[{name}] ep={epoch:4d} data={loss_data.item():.3e} phys={loss_phys.item():.3e} "
                    f"projL={loss_proj_cons.item():.3e} rollL={loss_rollout.item():.3e} "
                    f"lamP={lambda_proj:.3f} lamR={lambda_roll:.3f} valMAE={val_mae:.1f} valRMSE={val_rmse:.1f} "
                    f"foreJump={max_fore_jump:.1f} caps=({dy_cap:.1f},{growth_cap:.4f}) w[min,max]=[{wmin:.2f},{wmax:.2f}]"
                )

            if epoch >= cfg['min_epochs'] and no_improve >= cfg['early_stop_patience']:
                print(f"[{name}] early stop at epoch {epoch}")
                break

    if best_state is not None:
        model.load_state_dict(best_state)

    elapsed = time.time() - start
    y_train_pred = model.predict_itot_numpy(t_train)

    y_val_raw = model.predict_itot_numpy(t_val)
    if cfg.get('use_projection', False):
        y_val_pred = project_sequence(
            y_val_raw,
            y0=float(y_train[-1]),
            dy_cap=dy_cap,
            growth_cap=growth_cap,
            smooth_alpha=float(cfg.get('proj_smooth_alpha', 0.15)),
        )
    else:
        y_val_pred = y_val_raw

    out_chk = model.netf_data()
    residuals = [float(out_chk[i].abs().mean().detach().cpu().item()) for i in range(8)]

    y_fore_raw = model.predict_itot_numpy(t_future_eval_np)
    if cfg.get('use_projection', False):
        y_fore_final = project_sequence(
            y_fore_raw,
            y0=float(y_train[-1]),
            dy_cap=dy_cap,
            growth_cap=growth_cap,
            smooth_alpha=float(cfg.get('proj_smooth_alpha', 0.15)),
        )
    else:
        y_fore_final = y_fore_raw

    if len(y_fore_final) > 1:
        dy_fore_final = np.diff(y_fore_final)
        max_fore_jump = float(np.max(dy_fore_final))
        max_fore_growth = float(np.max(dy_fore_final / (y_fore_final[:-1] + 1.0)))
    else:
        max_fore_jump = 0.0
        max_fore_growth = 0.0

    result = {
        'idea': name,
        'use_projection': int(bool(cfg.get('use_projection', False))),
        'use_v2_rollout': int(bool(cfg.get('use_v2_rollout', False))),
        'train_rel_rmse': rel_rmse(y_train, y_train_pred),
        'val_mae_81_90': float(np.mean(np.abs(y_val - y_val_pred))),
        'val_rmse_81_90': float(np.sqrt(np.mean((y_val - y_val_pred) ** 2))),
        'val_rmse_raw_81_90': float(np.sqrt(np.mean((y_val - y_val_raw) ** 2))),
        'best_epoch': best_epoch,
        'train_time_sec': elapsed,
        'max_forecast_jump': max_fore_jump,
        'max_forecast_growth': max_fore_growth,
        'dy_cap': dy_cap,
        'growth_cap': growth_cap,
        'res_f1': residuals[0],
        'res_f2': residuals[1],
        'res_f3': residuals[2],
        'res_f4': residuals[3],
        'res_f5': residuals[4],
        'res_f6': residuals[5],
        'res_f7': residuals[6],
        'res_f8': residuals[7],
        'pred_val': y_val_pred,
    }

    print(
        f"[{name}] DONE | MAE={result['val_mae_81_90']:.2f}, RMSE={result['val_rmse_81_90']:.2f}, "
        f"rawRMSE={result['val_rmse_raw_81_90']:.2f}, best_epoch={best_epoch}, "
        f"foreJump={max_fore_jump:.1f}, proj={result['use_projection']}, v2={result['use_v2_rollout']}, time={elapsed:.1f}s"
    )
    return result


In [6]:
cfg_base = {
    'use_features': True,
    'adaptive_weights': False,
    'use_colloc': True,
    'use_mono': True,
    'use_projection': True,
    'use_v2_rollout': True,
    'proj_smooth_alpha': 0.15,
    'proj_smooth_alpha_train': 0.10,
    'data_loss': 'huber',
    'huber_beta': 0.45,
    'time_weight_start': 0.30,
    'lambda_data_rel': 0.08,
    'lambda_trend': 0.45,
    'trend_tail_k': 16,
    'trend_beta': 1200.0,
    'phys_balance': 'inv_softmax',
    'phys_ema_beta': 0.95,
    'phys_eps': 1e-8,
    'phys_weight_clip': 10.0,
    'phys_temp': 0.80,
    'lr': 4e-4,
    'epochs': 5000,
    'warmup': 1000,
    'ramp': 1800,
    'lambda_phys_max': 0.10,
    'lambda_mono': 0.05,
    'lambda_spike': 0.08,
    'lambda_growth': 0.14,
    'lambda_smooth': 0.05,
    'lambda_anchor': 0.15,
    'lambda_ic0': 0.00,
    'adaptive_boundary': False,
    'ab_alpha': 1.5,
    'ab_max': 3.0,
    'lambda_proj': 0.30,
    'proj_start_epoch': 300,
    'proj_ramp_epochs': 800,
    'lambda_roll': 0.18,
    'rollout_start_epoch': 1000,
    'rollout_ramp_epochs': 1000,
    'lambda_roll_dyn': 0.5,
    'rollout_steps': 5,
    'proj_horizon_min': 6,
    'proj_horizon_max': 14,
    'eval_horizon': 10,
    'spike_horizon': 10,
    'cap_tail_window': 20,
    'spike_cap_q_tail': 0.95,
    'spike_cap_mult_tail': 1.8,
    'growth_cap_q_tail': 0.95,
    'growth_cap_mult_tail': 2.0,
    'spike_cap_q_global': 0.80,
    'spike_cap_mult_global': 0.70,
    'growth_cap_q_global': 0.80,
    'growth_cap_mult_global': 0.40,
    'anchor_tail_window': 12,
    'best_spike_alpha': 0.08,
    'best_growth_alpha': 0.08,
    'n_colloc': 160,
    'eval_every': 100,
    'print_every': 200,
    'early_stop_patience': 1200,
    'min_epochs': 1800,
}

rows = []

res_base = run_experiment('V2_BASELINE', cfg_base, t_train, y_train, t_val, y_val)
rows.append({k: res_base[k] for k in res_base if k != 'pred_val'})

cfg_ca = dict(cfg_base)
cfg_ca.update({
    'best_select_start_epoch': cfg_base['warmup'],
    'best_raw_mix': 0.35,
    'proj_start_epoch': max(300, cfg_base['warmup'] - 200),
    'proj_ramp_epochs': 900,
    'rollout_start_epoch': cfg_base['warmup'] + 200,
    'rollout_ramp_epochs': 1200,
    'proj_horizon_min': 4,
    'proj_horizon_max': 12,
})
res_ca = run_experiment('CA_BESTFIX', cfg_ca, t_train, y_train, t_val, y_val)
rows.append({k: res_ca[k] for k in res_ca if k != 'pred_val'})

out_df = pd.DataFrame(rows)
os.makedirs('outputs', exist_ok=True)
out_path = f'outputs/{REGION}_ca_bestfix_compare.csv'
out_df.to_csv(out_path, index=False)
print('saved:', out_path)
out_df


[V2_BASELINE] ep=   0 data=1.108e+01 phys=3.589e-04 projL=1.722e+10 rollL=1.000e+00 lamP=0.000 lamR=0.000 valMAE=15248.4 valRMSE=17599.6 foreJump=0.0 caps=(3735.3,0.0437) w[min,max]=[0.00,7.99]
[V2_BASELINE] ep= 200 data=4.215e+00 phys=8.076e-03 projL=5.176e+00 rollL=1.521e+00 lamP=0.000 lamR=0.000 valMAE=15248.4 valRMSE=17599.6 foreJump=0.0 caps=(3735.3,0.0437) w[min,max]=[0.07,6.85]
[V2_BASELINE] ep= 400 data=1.344e+00 phys=4.056e-04 projL=1.160e-01 rollL=9.060e-01 lamP=0.037 lamR=0.000 valMAE=15248.4 valRMSE=17599.6 foreJump=0.0 caps=(3735.3,0.0437) w[min,max]=[0.00,7.99]
[V2_BASELINE] ep= 600 data=3.415e-02 phys=3.263e-04 projL=3.173e-05 rollL=9.814e-03 lamP=0.113 lamR=0.000 valMAE=2051.2 valRMSE=2121.7 foreJump=3175.0 caps=(3735.3,0.0437) w[min,max]=[0.00,8.00]
[V2_BASELINE] ep= 800 data=4.794e-03 phys=5.460e-04 projL=9.102e-06 rollL=2.359e-03 lamP=0.188 lamR=0.000 valMAE=426.4 valRMSE=493.8 foreJump=3175.0 caps=(3735.3,0.0437) w[min,max]=[0.00,7.99]
[V2_BASELINE] ep=1000 data=3.4

,idea,use_projection,use_v2_rollout,train_rel_rmse,val_mae_81_90,val_rmse_81_90,val_rmse_raw_81_90,best_epoch,train_time_sec,max_forecast_jump,...,dy_cap,growth_cap,res_f1,res_f2,res_f3,res_f4,res_f5,res_f6,res_f7,res_f8
0,V2_BASELINE,1,1,0.018236,426.442200,493.848572,3494.489502,800,25.271977,3174.984375,...,3735.269287,0.04365,0.032441,0.007251,0.124096,0.083695,0.047946,0.160122,0.041826,0.078079
1,CA_BESTFIX,1,1,0.015233,1437.623413,1479.079224,960.552734,3200,52.496433,3174.984375,...,3735.269287,0.04365,0.017865,0.000060,0.018611,0.005508,0.010788,0.024696,0.002387,0.073498
